# Поиск кандидатов для запросов в услугах Авито

## Задача

Пользователь вводит запрос, например «ремонт холодильников», в каком-то городе и иногда с фильтром вроде
«Вид услуги: Ремонт техники». Нужно для каждого из 2 452 запросов бенчмарка выбрать 50 объявлений из 189 тысяч
так, чтобы среди них оказались те, которые пользователь действительно выбрал. Качество — Recall@50: какая
доля правильных объявлений попала в эти 50. Порядок внутри пятидесяти не важен.

## Что в ней сложного

- **Объявления бенчмарка почти не встречаются в train** — общих всего 9.6%. Запомнить, что по запросу X
  выбирают объявление Y, не получится: объявления нужно находить по их содержанию.
- **Запросы короткие**, 2–3 слова, а у объявлений бывает общий заголовок («Бетонные работы»), и нужные слова
  есть только в описании.
- **Место решает много.** В 82% случаев выбранное объявление из той же локации, что и запрос, а остальные
  чаще всего из соседних городов. При этом у 17% запросов бенчмарка в корпусе вообще нет объявлений их
  локации.
- **Половина запросов новые.** 37% текстов бенчмарка встречались в train, остальные — нет.

## Как устроено решение

Поиск идёт в две ступени — так устроено большинство поисковых систем.

**Ступень 1: широкий отбор кандидатов.** Для запроса каждое объявление корпуса оценивается несколькими
простыми методами — по совпадению слов, по смыслу, по месту. По каждому методу берутся его лучшие объявления,
и всё это объединяется в пул примерно из 700 кандидатов на запрос. Методы ошибаются по-разному, поэтому
вместе они почти ничего не теряют: в пуле оказывается 94% правильных ответов.

**Ступень 2: выбор 50 из пула.** Здесь уже нужно учитывать все оценки вместе. Сначала — простая взвешенная
сумма оценок, веса которой подбираются на валидации. Затем поверх неё — модель CatBoost, обученная
ранжировать кандидатов: она учитывает сочетания оценок и свойства объявлений (цену, рейтинг, число отзывов).
В ответ идёт модель, которая лучше на валидации.

```
запрос ─┬─ BM25 (слова запроса в заголовке и описании) ──┐
        ├─ e5 (близость по смыслу) ───────────────────────┤
        ├─ символьные n-граммы (словоформы, опечатки) ────┤──► пул ~700 ──► взвешенная сумма ──► CatBoost ──► 50
        ├─ фильтры запроса («вид/тип услуги») ────────────┤
        ├─ микрокатегория похожих запросов из train ──────┤
        └─ место: та же локация, расстояние ──────────────┘
```

## Методы оценки объявления

1. **BM25** — совпадение слов запроса со словами заголовка, описания и «вида/типа услуги» объявления.
   Слова приводятся к начальной форме. Самый сильный метод.
2. **e5** — близость по смыслу. Готовая нейросеть `intfloat/multilingual-e5-base` (без дообучения)
   превращает тексты запроса и объявления в векторы; чем ближе векторы, тем ближе смысл. Находит объявления
   без общих с запросом слов: «отделочные работы» → «ремонт квартир под ключ».
3. **Символьные n-граммы** — совпадение кусочков слов заголовка (по 3–5 букв). Помогает с опечатками
   и формами слов, которые не распознал лемматизатор.
4. **Фильтры** — совпадение выбранного пользователем фильтра «вид/тип услуги» с тем же полем объявления.
5. **Микрокатегория** — какие микрокатегории выбирали люди по похожим запросам в train. Так знания из train
   переносятся на новые объявления — через категорию, а не через конкретные ID.
6. **Та же локация** — объявление из той же локации, что и запрос.
7. **Близость** — чем ближе объявление к центру локации запроса, тем выше оценка.

Как к этой схеме пришли и что не сработало — в `01_exploration` и `02_experiments`.

## Как запустить

1. Положить в папку `dataset/` в корне репозитория файлы `train.parquet`, `benchmark_queries.parquet`,
   `benchmark_items.parquet`.
2. Установить зависимости: `uv sync`.
3. Выполнить ноутбук целиком (Run All).

Модель e5 скачивается с Hugging Face автоматически при первом запуске (~1 ГБ, лицензия MIT), дальше всё
работает локально. Ответ сохраняется в `submissions/answer_NN.csv` со следующим свободным номером — прежние
файлы не перезаписываются.

**Время на GTX 1650 / 16 ГБ RAM.** Первый запуск — около полутора часов: кодирование текстов объявлений в e5
(~35 минут), лемматизация корпусов (~5 минут), подсчёт кандидатов (~25 минут), обучение CatBoost (~13 минут).
Промежуточные результаты сохраняются в `artifacts/cache`, повторный запуск занимает ~20 минут. Чтобы
пересчитать всё с нуля, достаточно удалить `artifacts/cache`.

**Воспроизводимость.** Все случайные шаги (разбиение данных, выбор запросов, CatBoost) используют
фиксированный `SEED`, число потоков зафиксировано, версии библиотек закреплены в `uv.lock`. Два полных прогона на одной
машине дали побайтово одинаковый файл ответа. На другом железе вычисления с плавающей точкой в нейросети могут
отличаться в последних знаках, и у объявлений с почти равной оценкой на границе топ-50 возможны единичные
перестановки. Чтобы это было видно, последняя ячейка сравнивает свежий ответ с отправленным файлом
(`submissions/submitted.csv`), если он есть.

## 1. Настройки

In [1]:
import csv
import json
import os
import re
import time
from functools import lru_cache
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import pymorphy2
import scipy.sparse as sp
import torch
from catboost import CatBoostRanker, Pool
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "dataset"
MODEL_DIR = ROOT / "artifacts" / "e5_baseline" / "multilingual-e5-base"
CACHE_DIR = ROOT / "artifacts" / "cache"
SUBMISSIONS_DIR = ROOT / "submissions"
for d in (CACHE_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
N_THREADS = 4
K = 50

torch.manual_seed(SEED)
torch.set_num_threads(N_THREADS)
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not (MODEL_DIR / "model.safetensors").exists():
    snapshot_download("intfloat/multilingual-e5-base", local_dir=MODEL_DIR,
                      allow_patterns=["*.json", "model.safetensors", "README.md"])

QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
              "search_infm_params_text", "search_category"]
ITEM_COLS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
             "item_location_id", "item_microcat_id", "item_latitude", "item_longitude",
             "item_price", "item_rating", "item_rating_reviews_count",
             "item_is_phone_hidden", "item_is_message_forbidden"]
FEATURES = ["bm", "e5", "tc", "sp", "mc", "loc", "geo"]
FEATURE_NAMES = ["BM25", "e5", "n-граммы", "фильтры", "микрокатегория", "та же локация", "близость"]

print("device:", DEVICE)

device: cuda


## 2. Данные

В train нет идентификатора запроса, поэтому запросом считается сочетание всех полей `search_*`: текст, локация,
фильтры, флаг доставки. «Маникюр» в Москве и «маникюр» в Казани — разные запросы с разными правильными
ответами.

Некоторые строки train повторяются целиком — тот же запрос и то же объявление. Это повторные клики, для
ответа важен только сам факт выбора, поэтому дубли удаляются.

Цена, рейтинг и координаты в файлах хранятся в виде `decimal.Decimal` и переводятся в обычные числа.

In [2]:
def clean_items(df):
    df = df.copy()
    df["item_id"] = df["item_id"].astype(str)
    for col in ["item_latitude", "item_longitude", "item_price", "item_rating", "item_rating_reviews_count"]:
        df[col] = pd.to_numeric(df[col].astype(object), errors="coerce").astype("float32")
    for col in ["item_title_raw", "item_description_raw", "item_infm_params_text"]:
        df[col] = df[col].fillna("").astype(str)
    return df.reset_index(drop=True)


train = pd.read_parquet(DATA_DIR / "train.parquet", columns=QUERY_COLS + ITEM_COLS)
n_raw = len(train)
train = train.drop_duplicates(subset=QUERY_COLS + ["item_id"]).reset_index(drop=True)
train["search_infm_params_text"] = train["search_infm_params_text"].fillna("")
train["item_id"] = train["item_id"].astype(str)

bench_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
bench_queries["query_id"] = bench_queries["query_id"].astype(str)
bench_queries["search_infm_params_text"] = bench_queries["search_infm_params_text"].fillna("")
bench_items = clean_items(pd.read_parquet(DATA_DIR / "benchmark_items.parquet", columns=ITEM_COLS))
train_items = clean_items(train[ITEM_COLS].drop_duplicates("item_id"))

print(f"train: {n_raw} -> {len(train)} строк после удаления дублей")
print(f"объявлений в train: {len(train_items)}, в корпусе бенчмарка: {len(bench_items)}")

train: 497673 -> 467054 строк после удаления дублей
объявлений в train: 344825, в корпусе бенчмарка: 189212


### Как проверяется качество

Правильных ответов для бенчмарка у нас нет, а его объявления почти не встречаются в train. Поэтому решение
проверяется на самом train: часть запросов откладывается, и для них ищутся объявления среди всех объявлений
train — ответы для них известны.

Train делится на три части по запросам:

- **история (81%)** — по ней ищутся похожие запросы для метода «микрокатегория»;
- **обучение CatBoost (9%)**;
- **валидация (10%)** — на ней считается качество и подбираются веса.

Историю нельзя смешивать с обучением CatBoost: иначе для обучающих запросов метод «микрокатегория» строился
бы по их же правильным ответам, и CatBoost научился бы ему слишком доверять.

Делятся именно запросы (сочетания полей), а не тексты. Поэтому часть текстов валидации встречается в истории —
как и в бенчмарке, где таких 37%.

Ещё две поправки, чтобы валидация была похожа на бенчмарк:

- в бенчмарке каждый текст встречается один раз, а в валидации популярный текст вроде «маникюр» повторяется
  в сотнях городов — без поправки качество определяли бы несколько популярных текстов. Поэтому на каждый текст
  берётся одно случайное сочетание полей;
- в валидации знакомых по истории текстов 70%, а в бенчмарке 37%. Поэтому при подсчёте Recall@50 знакомые
  запросы получают меньший вес, а новые — больший, чтобы итог соответствовал доле 37%.

In [3]:
def query_key(df):
    return df[QUERY_COLS].fillna("").astype(str).agg("\x1f".join, axis=1)


keys = query_key(train)
rest_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED).split(train, groups=keys))
rest = train.iloc[rest_idx].assign(query_id=keys.iloc[rest_idx].values).reset_index(drop=True)
hist_idx, rank_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED).split(rest, groups=rest["query_id"]))
history = rest.iloc[hist_idx].reset_index(drop=True)
rank_part = rest.iloc[rank_idx].reset_index(drop=True)
val_part = train.iloc[val_idx].assign(query_id=keys.iloc[val_idx].values).reset_index(drop=True)


def one_query_per_text(part, limit=None):
    gt = part.groupby("query_id")["item_id"].agg(set)
    queries = (
        part.drop_duplicates("query_id")
        .sample(frac=1.0, random_state=SEED)
        .drop_duplicates("search_query")
        .reset_index(drop=True)
    )
    if limit:
        queries = queries.head(limit)
    return queries, [gt[q] for q in queries["query_id"]]


val_queries, val_gt = one_query_per_text(val_part)
rank_queries, rank_gt = one_query_per_text(rank_part, limit=10_000)
print(f"история: {len(history)} строк, обучение CatBoost: {len(rank_queries)} запросов, "
      f"валидация: {len(val_queries)} запросов")

история: 377823 строк, обучение CatBoost: 10000 запросов, валидация: 15839 запросов


## 3. Подготовка текста

Чтобы «телевизоров» в запросе совпадало с «телевизор» в объявлении, слова приводятся к начальной форме
(лемматизатор pymorphy2). Перед этим текст переводится в нижний регистр, `ё` заменяется на `е`, знаки
препинания отбрасываются.

У объявления есть поле с параметрами — одна длинная строка: «Вид услуги Ремонт и отделка Тип услуги Двери
Место оказания услуг Волгоград График работы …». Для поиска полезны только вид и тип услуги: по ним же
пользователь фильтрует поиск. Остальное (адрес, график, цены) только добавляет шум, поэтому из строки
вырезаются только эти два значения.

In [4]:
TOKEN_RE = re.compile(r"[a-zа-я0-9]+")
MORPH = pymorphy2.MorphAnalyzer()

PARAM_KEYS = [
    "Вид услуги", "Тип услуги", "Место оказания услуг", "Тип стоимости", "Начальная цена",
    "Работа по договору", "Гарантия", "Бригада", "Время для связи", "График работы",
    "Выполняю заказы", "Опыт работы", "Где вы оказываете услуги", "Ваши клиенты",
    "Кто оказывает услуги", "Услуга", "Стоимость", "Название услуги", "Куда выезжаете",
    "Продолжительность", "Готов закупить материалы", "Время работы", "Год окончания",
    "Специальность", "Учебное учреждение", "Вид работ", "Тип работ", "Тип техники",
    "Вид техники", "Марка", "Предмет", "Формат занятий",
]
KEY_ALT = "|".join(re.escape(k) for k in sorted(PARAM_KEYS, key=len, reverse=True))
SERVICE_RE = re.compile(rf"(Вид услуги|Тип услуги)\s+(.*?)(?=\s+(?:{KEY_ALT})\b|$)")


def normalize(text):
    text = text.lower().replace("ё", "е").replace("\xa0", " ")
    return " ".join(TOKEN_RE.findall(text))


@lru_cache(maxsize=2_000_000)
def lemma(token):
    if len(token) < 3 or not token.isalpha():
        return token
    return MORPH.parse(token)[0].normal_form.replace("ё", "е")


def lemmatize(text):
    return " ".join(lemma(t) for t in normalize(text).split())


def service_params(text):
    return " ".join(m.group(2) for m in SERVICE_RE.finditer(text)) if text else ""


example = bench_items.iloc[0]
print(example["item_title_raw"], "->", lemmatize(example["item_title_raw"]))
print(example["item_infm_params_text"][:120], "->", service_params(example["item_infm_params_text"]))

Ремонт/выкуп компьют. и ноутбуков с выездом на дом -> ремонт выкуп компить и ноутбук с выезд на дом
Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от  -> Компьютерная помощь Компьютеры


Лемматизация всех описаний занимает несколько минут, поэтому результат сохраняется и при повторном запуске берётся из кэша.

In [5]:
def corpus_texts(items, name):
    path = CACHE_DIR / f"{name}_texts.parquet"
    if path.exists():
        cached = pd.read_parquet(path)
        if len(cached) == len(items) and (cached["item_id"].values == items["item_id"].values).all():
            return cached
    title = items["item_title_raw"]
    params = items["item_infm_params_text"].map(service_params)
    out = pd.DataFrame({
        "item_id": items["item_id"].values,
        "title_norm": title.map(normalize),
        "full_lem": (title + " " + items["item_description_raw"] + " " + params).map(lemmatize),
        "params_lem": params.map(lemmatize),
    })
    out.to_parquet(path)
    return out


def query_texts(queries):
    q = queries["search_query"].fillna("").astype(str)
    params = queries["search_infm_params_text"].fillna("").astype(str).map(service_params)
    return pd.DataFrame({
        "raw": q,
        "title_norm": q.map(normalize),
        "title_lem": q.map(lemmatize),
        "params_lem": params.map(lemmatize),
    })

## 4. Методы, основанные на словах

**BM25** — классическая формула поисковых систем. Объявление получает тем больше очков, чем больше в нём слов
запроса, причём редкие слова («холодильник») весят больше частых («услуги»). По сравнению с простым подсчётом
у BM25 два полезных свойства: десятое упоминание слова добавляет меньше, чем первое, а длинное описание не
выигрывает только за счёт длины. Сравниваются слова запроса со словами заголовка, описания и вида/типа услуги.

Формула записана через разреженные матрицы: веса слов в объявлениях считаются один раз, а оценки для сотни
запросов сразу получаются одним умножением матриц. Готовая библиотека `rank_bm25` тратит 0.09 секунды на
запрос — 25 минут на одну валидацию.

**Символьные n-граммы** сравнивают кусочки слов заголовка длиной 3–5 букв: у «телевизор» и «телевизора»
почти все такие кусочки общие, поэтому метод устойчив к формам слов и опечаткам.

**Фильтры** сравнивают вид/тип услуги из фильтра запроса с тем же полем объявления.

In [6]:
class BM25:
    def __init__(self, k1=1.2, b=0.75, min_df=2):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(min_df=min_df, token_pattern=r"\S+", dtype=np.float32)

    def fit(self, docs):
        tf = self.vec.fit_transform(docs).tocsr()
        n = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        idf = np.log1p((n - df + 0.5) / (df + 0.5)).astype(np.float32)
        doc_len = np.asarray(tf.sum(axis=1)).ravel()
        norm = self.k1 * (1 - self.b + self.b * doc_len / doc_len.mean())
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))
        data = tf.data * (self.k1 + 1) / (tf.data + norm[rows]) * idf[tf.indices]
        self.doc_weights = sp.csr_matrix((data.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape)
        return self

    def transform_queries(self, queries):
        q = self.vec.transform(queries).tocsr()
        q.data[:] = 1.0
        return q

## 5. Смысловая близость: e5

`intfloat/multilingual-e5-base` — готовая открытая нейросеть, обученная переводить тексты в векторы так, чтобы
близкие по смыслу тексты давали близкие векторы. Модель используется как есть, без дообучения. Авторы модели
требуют помечать запросы префиксом `query:`, а документы — `passage:`, так и сделано.

Объявление кодируется коротким текстом «заголовок. вид/тип услуги» (до 64 токенов). С описанием кодирование
всех объявлений на GTX 1650 заняло бы около 5 часов, а слова описания и так учитывает BM25.

40% заголовков повторяются, поэтому каждый уникальный текст кодируется один раз, а результат сохраняется на
диск частями — прерванный расчёт продолжается с места остановки. Тексты кодируются в фиксированном порядке
(по длине, затем по алфавиту), чтобы разбиение на пачки, а значит и результат, не менялся от запуска к запуску.

In [7]:
E5_CACHE = CACHE_DIR / "e5_text_cache"
_e5_model = None


def e5_model():
    global _e5_model
    if _e5_model is None:
        _e5_model = SentenceTransformer(str(MODEL_DIR), device=DEVICE)
        _e5_model.max_seq_length = 64
    return _e5_model


def load_e5_cache():
    texts, embs = [], []
    for f in sorted(E5_CACHE.glob("part_*.parquet")):
        texts += pd.read_parquet(f)["text"].tolist()
        embs.append(np.load(f.with_suffix(".npy")))
    return texts, (np.concatenate(embs) if embs else np.zeros((0, 768), np.float32))


def e5_items(items, chunk=20_000):
    params = items["item_infm_params_text"].map(service_params)
    texts = (items["item_title_raw"].str.strip() + ". " + params).str.lower()
    E5_CACHE.mkdir(parents=True, exist_ok=True)
    have, _ = load_e5_cache()
    todo = sorted(set(texts) - set(have), key=lambda t: (len(t), t))
    n_parts = len(list(E5_CACHE.glob("part_*.parquet")))
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]
        vecs = e5_model().encode(["passage: " + t for t in part], batch_size=128,
                                 normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        name = E5_CACHE / f"part_{n_parts:04d}"
        np.save(name.with_suffix(".npy"), vecs.astype(np.float32))
        pd.DataFrame({"text": part}).to_parquet(name.with_suffix(".parquet"))
        n_parts += 1
    have, emb = load_e5_cache()
    pos = pd.Series(np.arange(len(have)), index=have)
    pos = pos[~pos.index.duplicated()]
    return emb[pos.reindex(texts).to_numpy()]


def e5_queries(texts):
    return e5_model().encode(["query: " + t for t in texts], batch_size=256, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

Индекс корпуса собирает всё нужное для быстрых оценок: матрицы слов для BM25, n-грамм и фильтров и векторы
e5 всех объявлений (они целиком лежат в памяти видеокарты).

In [8]:
def build_index(items, name):
    t = corpus_texts(items, name)
    tc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, dtype=np.float32)
    spv = TfidfVectorizer(token_pattern=r"\S+", sublinear_tf=True, dtype=np.float32)
    bm = BM25().fit(t["full_lem"])
    return {
        "tc": tc, "D_tc": tc.fit_transform(t["title_norm"]).T.tocsr(),
        "sp": spv, "D_sp": spv.fit_transform(t["params_lem"]).T.tocsr(),
        "bm": bm, "D_bm": bm.doc_weights.T.tocsr(),
        "E": torch.from_numpy(e5_items(items)).to(DEVICE),
    }


def encode_queries(index, queries):
    qt = query_texts(queries)
    return {
        "tc": index["tc"].transform(qt["title_norm"]).tocsr(),
        "sp": index["sp"].transform(qt["params_lem"]).tocsr(),
        "bm": index["bm"].transform_queries(qt["title_lem"]),
        "e5": torch.from_numpy(e5_queries(qt["raw"].tolist())),
    }

## 6. Микрокатегория по похожим запросам

Объявления бенчмарка почти не встречаются в train, но их микрокатегории — те же. Поэтому из train можно
узнать, **какого рода** объявления выбирают по запросу, даже если конкретных объявлений там нет.

Для запроса находятся 30 самых похожих по написанию запросов из истории и смотрится, в каких микрокатегориях
были объявления, которые по ним выбрали. Чем похожее запрос, тем больше его голос. Если такой же текст уже
встречался в истории (как у 37% запросов бенчмарка), его голос почти полностью определяет результат.
Объявление получает оценку — долю выборов, пришедшуюся на его микрокатегорию.

Заодно запоминается, насколько похож самый близкий запрос из истории: для CatBoost это сигнал, можно ли
доверять этой оценке.

In [9]:
class MicrocatPrior:
    def __init__(self, pairs, k=30, power=2.0):
        self.k, self.power = k, power
        texts = pairs["search_query"].astype(str).map(normalize).astype("category")
        codes = texts.cat.codes.to_numpy()
        self.mcats = np.sort(pairs["item_microcat_id"].unique())
        self.mc_index = {m: i for i, m in enumerate(self.mcats)}
        counts = sp.csr_matrix(
            (np.ones(len(pairs), np.float32), (codes, pairs["item_microcat_id"].map(self.mc_index))),
            shape=(len(texts.cat.categories), len(self.mcats)),
        )
        rs = np.asarray(counts.sum(1)).ravel()
        self.dist = sp.diags(1.0 / np.maximum(rs, 1e-9)).dot(counts).tocsr().astype(np.float32)
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                   sublinear_tf=True, dtype=np.float32)
        self.T = self.vec.fit_transform(texts.cat.categories).T.tocsr()

    def predict(self, queries, batch=1024):
        Q = self.vec.transform(queries["search_query"].astype(str).map(normalize)).tocsr()
        probs = np.zeros((Q.shape[0], len(self.mcats) + 1), np.float32)
        nearest = np.zeros(Q.shape[0], np.float32)
        for s in range(0, Q.shape[0], batch):
            sim = torch.from_numpy((Q[s:s + batch] @ self.T).toarray())
            val, idx = sim.topk(self.k, dim=1)
            nearest[s:s + batch] = val[:, 0].numpy()
            w = val.clamp_min(0) ** self.power
            w = w / w.sum(1, keepdim=True).clamp_min(1e-9)
            rows = np.repeat(np.arange(len(idx)), self.k)
            W = sp.csr_matrix((w.numpy().ravel(), (rows, idx.numpy().ravel())), shape=(len(idx), self.T.shape[1]))
            probs[s:s + batch, :-1] = (W @ self.dist).toarray()
        return probs, nearest

    def item_columns(self, items):
        unknown = len(self.mcats)
        return np.array([self.mc_index.get(m, unknown) for m in items["item_microcat_id"]])

## 7. Место

82% выбранных объявлений — из той же локации, что и запрос. Но фильтровать по городу нельзя: остальные 18%
правильных ответов потерялись бы навсегда, а это чаще всего исполнители из соседних городов. Поэтому место
учитывается двумя мягкими оценками: совпадение локации и близость в километрах.

У запроса есть только номер локации, без координат. Центром локации считается медиана координат её
объявлений — в том числе объявлений train: у 17% запросов бенчмарка в его корпусе нет объявлений их локации.
Расстояние — по поверхности Земли от объявления до этого центра.

In [10]:
def location_centroids(*item_frames):
    items = pd.concat([f[["item_location_id", "item_latitude", "item_longitude"]] for f in item_frames])
    return items.dropna().groupby("item_location_id")[["item_latitude", "item_longitude"]].median()


def item_coords(items, centroids):
    lat = items["item_latitude"].to_numpy(np.float64)
    lon = items["item_longitude"].to_numpy(np.float64)
    bad = np.isnan(lat) | np.isnan(lon)
    c = centroids.reindex(items.loc[bad, "item_location_id"])
    lat[bad], lon[bad] = c["item_latitude"].to_numpy(), c["item_longitude"].to_numpy()
    return torch.from_numpy(np.radians(np.stack([lat, lon], 1)).astype(np.float32))


def query_coords(queries, centroids):
    c = centroids.reindex(queries["search_location_id"].to_numpy())
    return torch.from_numpy(np.radians(c[["item_latitude", "item_longitude"]].to_numpy(np.float64)).astype(np.float32))


def haversine_km(q, d):
    dlat = d[None, :, 0] - q[:, None, 0]
    dlon = d[None, :, 1] - q[:, None, 1]
    a = torch.sin(dlat / 2) ** 2 + torch.cos(q[:, None, 0]) * torch.cos(d[None, :, 0]) * torch.sin(dlon / 2) ** 2
    km = 2 * 6371.0 * torch.asin(torch.sqrt(a.clamp(0, 1)))
    return torch.nan_to_num(km, nan=20000.0)

## 8. Ступень 1: пул кандидатов

Для пачки запросов все методы оценивают сразу все объявления корпуса. Оценки разных методов живут в разных
шкалах: у BM25 они зависят от длины запроса, а оценки e5 почти у всех объявлений лежат между 0.75 и 0.93.
Чтобы их можно было складывать и сравнивать, внутри каждого запроса оценки текстовых методов и микрокатегории
приводятся к шкале от 0 до 1 (худшее объявление корпуса — 0, лучшее — 1).

Пул кандидатов для запроса — объединение:

- 100 лучших объявлений по каждому методу из BM25, e5, n-грамм и фильтров — по всему корпусу;
- 100 лучших по каждому из них среди объявлений той же локации;
- 300 лучших по простой сумме всех оценок.

Для кандидатов сохраняются все оценки и расстояние в километрах. Дальше работа идёт только с пулом, поэтому
его подсчёт — самый долгий шаг — выполняется один раз и сохраняется на диск.

In [11]:
def minmax(x):
    lo, hi = x.amin(1, keepdim=True), x.amax(1, keepdim=True)
    return (x - lo) / (hi - lo).clamp_min(1e-6)


def query_features(index, Q, rows, ctx):
    return {
        "bm": minmax(torch.from_numpy((Q["bm"][rows] @ index["D_bm"]).toarray())),
        "tc": minmax(torch.from_numpy((Q["tc"][rows] @ index["D_tc"]).toarray())),
        "sp": minmax(torch.from_numpy((Q["sp"][rows] @ index["D_sp"]).toarray())),
        "e5": minmax((Q["e5"][rows].to(DEVICE) @ index["E"].T).cpu()),
        "mc": minmax(ctx["probs"][rows][:, ctx["item_mc"]]),
        "loc": (ctx["item_loc"][None, :] == ctx["q_loc"][rows, None]).float(),
        "dist": haversine_km(ctx["q_xy"][rows], ctx["item_xy"]),
    }


def candidate_ids(f, n_each=100, n_sum=300):
    near = f["loc"] + torch.exp(-f["dist"] / 300)
    total = f["bm"] + f["e5"] + f["tc"] + f["sp"] + f["mc"] + near
    ids = [total.topk(n_sum, dim=1).indices]
    for name in ["bm", "e5", "tc", "sp"]:
        ids.append(f[name].topk(n_each, dim=1).indices)
        ids.append((f[name] + 10 * f["loc"]).topk(n_each, dim=1).indices)
    return torch.cat(ids, 1)


RAW = ["bm", "e5", "tc", "sp", "mc", "loc", "dist"]


class Candidates:
    def __init__(self, qidx, doc, X, n_queries, nearest):
        counts = np.bincount(qidx, minlength=n_queries)
        pos = np.arange(len(qidx)) - np.repeat(np.cumsum(counts) - counts, counts)
        self.X = torch.zeros((n_queries, counts.max(), len(RAW)))
        self.X[qidx, pos] = torch.from_numpy(X)
        self.doc = np.full((n_queries, counts.max()), -1, dtype=np.int64)
        self.doc[qidx, pos] = doc
        self.valid = torch.from_numpy(self.doc >= 0)
        self.nearest = nearest

    def features(self, tau):
        F = self.X.clone()
        F[..., -1] = torch.exp(-F[..., -1] / tau)
        return F

    def linear_score(self, w):
        F = self.features(w["tau"])
        s = sum(w[name] * F[..., i] for i, name in enumerate(FEATURES))
        return s.masked_fill(~self.valid, -1e9)

    def relevance(self, item_ids, gt):
        ids = np.where(self.doc >= 0, item_ids[np.clip(self.doc, 0, None)], "")
        return torch.from_numpy(np.array([np.isin(row, list(g)) for row, g in zip(ids, gt)]))


def collect(queries, items, index, prior, centroids, name, batch=128):
    path = CACHE_DIR / f"candidates_{name}.npz"
    if path.exists():
        z = np.load(path)
        if int(z["n_queries"]) == len(queries):
            return Candidates(z["qidx"], z["doc"], z["X"], len(queries), z["nearest"])
    Q = encode_queries(index, queries)
    probs, nearest = prior.predict(queries)
    ctx = {
        "probs": torch.from_numpy(probs),
        "item_mc": torch.from_numpy(prior.item_columns(items)),
        "item_loc": torch.from_numpy(items["item_location_id"].to_numpy(np.int64)),
        "q_loc": torch.from_numpy(queries["search_location_id"].to_numpy(np.int64)),
        "item_xy": item_coords(items, centroids),
        "q_xy": query_coords(queries, centroids),
    }
    qidx, docs, feats = [], [], []
    for start in range(0, len(queries), batch):
        rows = slice(start, min(start + batch, len(queries)))
        f = query_features(index, Q, rows, ctx)
        cand = candidate_ids(f)
        for i in range(cand.shape[0]):
            d = torch.unique(cand[i])
            qidx.append(np.full(len(d), start + i, dtype=np.int32))
            docs.append(d.numpy())
            feats.append(torch.stack([f[name][i, d] for name in RAW], 1).numpy())
    qidx, docs, feats = np.concatenate(qidx), np.concatenate(docs), np.concatenate(feats)
    np.savez(path, qidx=qidx, doc=docs, X=feats, nearest=nearest, n_queries=len(queries))
    return Candidates(qidx, docs, feats, len(queries), nearest)

Пулы для валидации и для обучения CatBoost собираются по объявлениям train, микрокатегория — по истории.

In [12]:
t0 = time.time()
index = build_index(train_items, "local")
prior = MicrocatPrior(history)
centroids = location_centroids(train_items)
val_cand = collect(val_queries, train_items, index, prior, centroids, "val")
rank_cand = collect(rank_queries, train_items, index, prior, centroids, "rank")
del index
print(f"{time.time() - t0:.0f} с, кандидатов на запрос: {val_cand.valid.sum(1).float().mean():.0f}")

72 с, кандидатов на запрос: 697


## 9. Ступень 2а: взвешенная сумма оценок

Самый простой способ выбрать 50 из пула — сложить оценки всех методов с весами и взять 50 лучших:

```
итог = w₁·BM25 + w₂·e5 + w₃·n-граммы + w₄·фильтры + w₅·микрокатегория + w₆·та_же_локация + w₇·exp(−км / τ)
```

Последнее слагаемое — оценка близости: 1 для объявления в центре локации и всё меньше с расстоянием; τ задаёт,
как быстро она убывает.

Качество считается как Recall@50: для каждого запроса — доля его правильных объявлений среди выбранных 50,
затем среднее по запросам (с поправкой на долю знакомых текстов, см. раздел 2). Сначала — сколько правильных
ответов вообще есть в пуле: выше этого значения никакой выбор из пула не поднимется.

In [13]:
item_ids = train_items["item_id"].to_numpy()
val_rel = val_cand.relevance(item_ids, val_gt)
val_n_rel = torch.tensor([len(g) for g in val_gt], dtype=torch.float32)

seen_texts = set(history["search_query"].astype(str).map(normalize))
val_seen = val_queries["search_query"].astype(str).map(normalize).isin(seen_texts).to_numpy()
val_weight = torch.from_numpy(np.where(val_seen, 0.37 / val_seen.mean(), 0.63 / (1 - val_seen.mean())))
val_weight = (val_weight / val_weight.sum()).float()


def recall(score, rel, n_rel, weight, k=K):
    top = score.topk(k, dim=1).indices
    per_query = torch.gather(rel, 1, top).sum(1).float() / n_rel
    return float((per_query * weight).sum()), per_query


pool_recall = float((val_rel.sum(1).float() / val_n_rel * val_weight).sum())
print(f"правильных ответов в пуле кандидатов: {pool_recall:.4f}")

правильных ответов в пуле кандидатов: 0.9428


### Как подбираются веса

Веса подбираются перебором прямо по Recall@50. Начинаем со всех весов, равных 1. Берём первый вес, пробуем для
него значения из сетки (0, 0.25, 0.5, … 3), остальные не трогаем, и оставляем то, при котором Recall@50 выше.
Так по очереди для каждого веса и для τ. Потом весь проход повторяется, пока метрика растёт.

Почему не градиентный спуск или логистическая регрессия: Recall@50 меняется скачками (объявление либо попало
в 50, либо нет), градиента у неё нет. Логистическую регрессию пробовали отдельно — она оптимизирует
другое — угадывание каждого кандидата по отдельности — и на проверке проиграла перебору 0.025–0.03. А перебор здесь дешёвый:
оценки для пула уже посчитаны, одна проверка набора весов занимает доли секунды.

In [14]:
GRID = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3]
TAU_GRID = [100, 200, 300, 500, 1000]


def val_recall(w):
    return recall(val_cand.linear_score(w), val_rel, val_n_rel, val_weight)[0]


weights = {name: 1.0 for name in FEATURES} | {"tau": 300}
best = val_recall(weights)
for round_ in range(4):
    improved = False
    for name, grid in [(n, GRID) for n in FEATURES] + [("tau", TAU_GRID)]:
        for value in grid:
            r = val_recall(weights | {name: value})
            if r > best + 1e-5:
                best, weights, improved = r, weights | {name: value}, True
    print(f"проход {round_ + 1}: Recall@50 = {best:.4f}")
    if not improved:
        break
print(weights)

проход 1: Recall@50 = 0.8313


проход 2: Recall@50 = 0.8376


проход 3: Recall@50 = 0.8376
{'bm': 1.5, 'e5': 1, 'tc': 0.25, 'sp': 0.75, 'mc': 0.25, 'loc': 0.75, 'geo': 2, 'tau': 500}


**Не подогнаны ли веса под случайности выборки?** Проверка: валидация делится пополам, веса подбираются на
одной половине и проверяются на другой. Если бы перебор запоминал шум, на чужой половине качество заметно
падало бы. Заодно видно, насколько сами половины отличаются друг от друга — это точность нашей оценки:
разницу между решениями меньше неё нельзя считать улучшением.

In [15]:
half = torch.from_numpy(np.random.default_rng(SEED).random(len(val_queries)) < 0.5)
for fit_part, check_part in [(half, ~half), (~half, half)]:
    w_fit, fit_weight = dict(weights), val_weight * fit_part
    for name, grid in [(n, GRID) for n in FEATURES] + [("tau", TAU_GRID)]:
        scores = [recall(val_cand.linear_score(w_fit | {name: v}), val_rel, val_n_rel, fit_weight)[0] for v in grid]
        w_fit[name] = grid[int(np.argmax(scores))]
    check_weight = val_weight * check_part
    held = recall(val_cand.linear_score(w_fit), val_rel, val_n_rel, check_weight)[0] / float(check_weight.sum())
    full = recall(val_cand.linear_score(weights), val_rel, val_n_rel, check_weight)[0] / float(check_weight.sum())
    print(f"веса, подобранные на другой половине: {held:.4f}, итоговые веса: {full:.4f}")

веса, подобранные на другой половине: 0.8332, итоговые веса: 0.8372


веса, подобранные на другой половине: 0.8380, итоговые веса: 0.8380


**Результат и вклад методов.** Вклад метода — насколько падает Recall@50, если его убрать. Так видно, на чём
держится решение: веса сами по себе важность не показывают, потому что оценки методов по-разному распределены.

In [16]:
linear_scores = val_cand.linear_score(weights)
linear_r50, linear_per_query = recall(linear_scores, val_rel, val_n_rel, val_weight)
for k in (50, 100, 200):
    print(f"Recall@{k}: {recall(linear_scores, val_rel, val_n_rel, val_weight, k)[0]:.4f}")
print(f"знакомые тексты: {linear_per_query[val_seen].mean():.4f}, новые: {linear_per_query[~val_seen].mean():.4f}")

pd.DataFrame(
    {"вес": [weights[n] for n in FEATURES],
     "Recall@50 без метода": [round(val_recall(weights | {n: 0.0}), 4) for n in FEATURES]},
    index=FEATURE_NAMES,
)

Recall@50: 0.8376
Recall@100: 0.8795


Recall@200: 0.9120
знакомые тексты: 0.8502, новые: 0.8302


,вес,Recall@50 без метода
BM25,1.50,0.7468
e5,1.00,0.8229
n-граммы,0.25,0.8368
фильтры,0.75,0.8258
микрокатегория,0.25,0.8270
та же локация,0.75,0.8028
близость,2.00,0.7313


## 10. Ступень 2б: CatBoost

У взвешенной суммы два ограничения. Веса одинаковы для всех запросов, хотя, например, для знакомого запроса
микрокатегории можно доверять больше, чем для нового. И она не умеет пользоваться свойствами объявления, у
которых нет пары в запросе: ценой, рейтингом, числом отзывов.

CatBoostRanker — градиентный бустинг, обученный именно ранжировать: функция потерь YetiRank штрафует за
неправильный порядок кандидатов внутри одного запроса. Он переупорядочивает 200 лучших по взвешенной сумме
(в них 91% правильных ответов). Что он видит о каждом кандидате:

- оценки всех семи методов, расстояние в километрах и итог взвешенной суммы;
- место кандидата внутри запроса по BM25, e5, n-граммам, фильтрам и взвешенной сумме;
- об объявлении: цена, рейтинг, число отзывов, скрыт ли телефон, запрещены ли сообщения, длина описания,
  сколько в корпусе объявлений с таким же заголовком;
- о запросе: сколько в нём слов, выбраны ли фильтры, насколько похож самый близкий запрос из истории.

Обучается на отдельной части train (раздел 2), проверяется на той же валидации, что и взвешенная сумма.

In [17]:
N_RERANK = 200
ITEM_FEATURES = ["log_price", "rating", "log_reviews", "phone_hidden", "messages_off", "log_desc_len", "log_title_dups"]


def item_table(items):
    title_count = items.groupby(items["item_title_raw"].str.lower())["item_id"].transform("size")
    return np.column_stack([
        np.log1p(items["item_price"].fillna(0).clip(lower=0)),
        items["item_rating"].fillna(0),
        np.log1p(items["item_rating_reviews_count"].fillna(0)),
        items["item_is_phone_hidden"].fillna(False).astype(float),
        items["item_is_message_forbidden"].fillna(False).astype(float),
        np.log1p(items["item_description_raw"].str.len()),
        np.log1p(title_count),
    ]).astype(np.float32)


def rerank_frame(cand, queries, item_tab):
    score = cand.linear_score(weights)
    top = score.topk(N_RERANK, dim=1).indices
    F = torch.gather(cand.features(weights["tau"]), 1, top[..., None].expand(-1, -1, len(RAW)))
    km = torch.gather(cand.X[..., -1], 1, top)
    lin = torch.gather(score, 1, top)
    ranks = torch.stack([F[..., i] for i in range(4)] + [lin], -1).argsort(1, descending=True).argsort(1).float()
    doc = np.take_along_axis(cand.doc, top.numpy(), 1)
    q_tab = np.column_stack([
        queries["search_query"].astype(str).str.split().str.len(),
        (queries["search_infm_params_text"] != "").astype(float),
        cand.nearest,
    ]).astype(np.float32)
    X = np.concatenate([
        F.numpy(), km[..., None].numpy(), lin[..., None].numpy(), ranks.numpy(),
        item_tab[np.clip(doc, 0, None)],
        np.repeat(q_tab[:, None, :], N_RERANK, axis=1),
    ], -1)
    columns = (FEATURES + ["km", "linear"] + [f"rank_{n}" for n in ["bm", "e5", "tc", "sp", "linear"]]
               + ITEM_FEATURES + ["query_words", "has_filters", "nearest_query"])
    valid = doc >= 0
    return pd.DataFrame(X[valid], columns=columns), np.repeat(np.arange(len(queries)), N_RERANK)[valid.ravel()], top, valid


train_tab = item_table(train_items)
rank_rel = rank_cand.relevance(item_ids, rank_gt)
X_rank, g_rank, top_rank, valid_rank = rerank_frame(rank_cand, rank_queries, train_tab)
y_rank = torch.gather(rank_rel, 1, top_rank).numpy()[valid_rank].astype(float)
X_val, g_val, top_val, valid_val = rerank_frame(val_cand, val_queries, train_tab)
print(f"обучение: {len(X_rank)} строк, {y_rank.sum():.0f} правильных; валидация: {len(X_val)} строк")

обучение: 2000000 строк, 10290 правильных; валидация: 3167800 строк


In [18]:
t0 = time.time()
ranker = CatBoostRanker(loss_function="YetiRank", iterations=600, learning_rate=0.08, depth=6,
                        random_seed=SEED, thread_count=N_THREADS, verbose=200)
ranker.fit(Pool(X_rank, label=y_rank, group_id=g_rank))
ranker.save_model(str(CACHE_DIR / "catboost_ranker.cbm"))
(CACHE_DIR / "linear_weights.json").write_text(json.dumps(weights, indent=1))
print(f"{time.time() - t0:.0f} с")


def rerank_scores(model, X, valid):
    s = np.full(valid.shape, -1e9, dtype=np.float32)
    s[valid] = model.predict(X)
    return torch.from_numpy(s)


cb_scores = rerank_scores(ranker, X_val, valid_val)
cb_r50, cb_per_query = recall(cb_scores, torch.gather(val_rel, 1, top_val), val_n_rel, val_weight)
print(f"Recall@50: взвешенная сумма {linear_r50:.4f}, CatBoost {cb_r50:.4f}")
print(f"CatBoost — знакомые тексты: {cb_per_query[val_seen].mean():.4f}, новые: {cb_per_query[~val_seen].mean():.4f}")

0:	total: 1.47s	remaining: 14m 42s


200:	total: 4m 12s	remaining: 8m 21s


400:	total: 8m 26s	remaining: 4m 11s


599:	total: 12m 40s	remaining: 0us
762 с


Recall@50: взвешенная сумма 0.8376, CatBoost 0.8448
CatBoost — знакомые тексты: 0.8601, новые: 0.8358


На что CatBoost опирается больше всего (доля влияния на его оценки, %):

In [19]:
pd.Series(ranker.get_feature_importance(type="PredictionValuesChange"), index=X_rank.columns) \
    .sort_values(ascending=False).head(12).round(1)

mc                8.1
km                8.0
bm                7.5
linear            7.4
rank_linear       7.2
log_desc_len      5.8
log_reviews       5.5
rank_bm           5.3
rank_sp           5.0
e5                4.9
log_title_dups    4.5
geo               4.4
dtype: float64

Точность оценки — около ±0.005 (раздел 9). CatBoost идёт в ответ, только если он лучше взвешенной суммы
больше чем на эту величину; иначе предпочтение отдаётся более простой модели.

In [20]:
USE_CATBOOST = cb_r50 > linear_r50 + 0.005
print("для ответа:", "CatBoost" if USE_CATBOOST else "взвешенная сумма")

для ответа: CatBoost


## 11. Ответ

Для запросов бенчмарка всё повторяется на его корпусе: пул кандидатов, взвешенная сумма, CatBoost. Похожие
запросы ищутся уже по всему train, центры локаций — по объявлениям бенчмарка и train.

In [21]:
t0 = time.time()
index = build_index(bench_items, "bench")
bench_cand = collect(bench_queries, bench_items, index, MicrocatPrior(train),
                     location_centroids(bench_items, train_items), "bench")
del index

if USE_CATBOOST:
    X_b, _, top_b, valid_b = rerank_frame(bench_cand, bench_queries, item_table(bench_items))
    order = rerank_scores(ranker, X_b, valid_b).topk(K, dim=1).indices
    bench_top = np.take_along_axis(np.take_along_axis(bench_cand.doc, top_b.numpy(), 1), order.numpy(), 1)
else:
    order = bench_cand.linear_score(weights).topk(K, dim=1).indices
    bench_top = np.take_along_axis(bench_cand.doc, order.numpy(), 1)
print(f"{time.time() - t0:.0f} с")

46 с


Формат файла: колонки `query_id` и `answer` через запятую, 50 `item_id` внутри `answer` — через пробел.
ID записываются как строки, без перевода в числа, поэтому ведущие нули и регистр сохраняются. Файл получает
следующий свободный номер в `submissions/`, после записи проверяется на все требования задания.

In [22]:
version = 1
while (SUBMISSIONS_DIR / f"answer_{version:02d}.csv").exists():
    version += 1
answer_path = SUBMISSIONS_DIR / f"answer_{version:02d}.csv"

bench_ids = bench_items["item_id"].to_numpy()
with open(answer_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["query_id", "answer"])
    for qid, docs in zip(bench_queries["query_id"], bench_top):
        writer.writerow([qid, " ".join(bench_ids[docs[docs >= 0]])])

answer = pd.read_csv(answer_path, dtype=str, keep_default_na=False)
lists = answer["answer"].str.split(" ")
valid_ids = set(bench_ids)
assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique and set(answer["query_id"]) == set(bench_queries["query_id"])
assert lists.map(len).between(1, K).all() and lists.map(lambda x: len(set(x)) == len(x)).all()
assert lists.map(lambda x: all(i in valid_ids for i in x)).all()
print(f"{answer_path.relative_to(ROOT)}: {len(answer)} строк, формат корректен")
answer.head()

submissions\answer_01.csv: 2452 строк, формат корректен

,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf d722bcda1a555091 b92ee8f432ce...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 8703b2a0442c...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b d8fce513e4f000a7 d62074dd39ca...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 67b2cd847669eb36 a846a2a5241e...
4,YgHcM9MVbxKnxD1e,1a62634394544781 ba78ad593ec3412d 9c41fe789f98...


Сравнение с отправленным файлом: если положить его в `submissions/submitted.csv`, ячейка покажет, какая доля
объявлений в ответах совпала. На том же железе ожидается 100%.

In [23]:
submitted_path = SUBMISSIONS_DIR / "submitted.csv"
if submitted_path.exists():
    submitted = pd.read_csv(submitted_path, dtype=str, keep_default_na=False).set_index("query_id")["answer"]
    fresh = answer.set_index("query_id")["answer"]
    same = [len(set(fresh[q].split()) & set(submitted[q].split())) / K for q in fresh.index]
    print(f"совпадает {np.mean(same):.2%} объявлений, полностью одинаковых строк: {np.mean(np.array(same) == 1):.2%}")
else:
    print("submissions/submitted.csv нет — сравнивать не с чем")

submissions/submitted.csv нет — сравнивать не с чем


## 12. Итоги

Результат на валидации (доля знакомых текстов приведена к 37%, как в бенчмарке):

In [24]:
pd.DataFrame({"Recall@50": [pool_recall, linear_r50, cb_r50]},
             index=["правильных ответов в пуле кандидатов (потолок)", "взвешенная сумма", "CatBoost"]).round(4)

,Recall@50
правильных ответов в пуле кандидатов (потолок),0.9428
взвешенная сумма,0.8376
CatBoost,0.8448


**На чём держится решение.** Главные методы — BM25 по заголовку вместе с описанием и близость: без любого из
них Recall@50 падает примерно на 0.09–0.1. Совпадение локации добавляет ещё около 0.035. Смысловая близость
e5, фильтры и микрокатегория добавляют по 0.01–0.015 каждый. Символьные n-граммы почти ничего не дают поверх
лемматизации, но и не мешают.

**Какие ошибки находили и что с ними сделали** (подробно — в `02_experiments`):

| ошибка | пример | что сделано |
|---|---|---|
| разные формы слова | «телевизор» и «телевизора» | лемматизация, символьные n-граммы |
| короткий общий заголовок у правильного объявления | «Бетонные работы» на «заливка бетонных дорожек» | BM25 смотрит и в описание |
| синонимы без общих слов | «отделочные работы» → «Ремонт квартир и офисов под ключ» | e5 |
| общие слова дают ложное сходство | «услуги», «ГОСТ» | BM25 (редкие слова весят больше), фильтры |
| правильное объявление в соседнем городе | выезд мастера | мягкий бонус за место вместо фильтра, оценка близости |
| в корпусе нет объявлений локации запроса | 17% запросов бенчмарка | центры локаций и по объявлениям train |
| объявление-двойник в другом городе | одинаковые «Аренда прицепа…» в разных городах | не исправляется: так устроены данные, 40% заголовков повторяются |

**Что пробовали и отвергли** (отдельные проверки на той же валидации, на предыдущей версии решения
с Recall@50 = 0.836): набирать 50 «лучших по каждому методу» с квотами вместо суммы (0.746–0.764), объединять
ранги методов без весов — Reciprocal Rank Fusion (0.758), учить веса логистической регрессией (на 0.025–0.03
хуже перебора), оценку по «вектору истории» похожих запросов — её вклад полностью покрывают BM25 и e5. Объединение лучших по каждому методу хорошо для отбора кандидатов (ступень 1), но финальные 50
лучше выбирать моделью, которая видит все оценки сразу.

**Ограничения.** Валидация идёт по объявлениям train (345 тысяч), корпус бенчмарка меньше (189 тысяч), поэтому
цифры на лидерборде будут другими. e5 видит только заголовок и вид/тип услуги.

**Что можно улучшить.** Дообучить e5 на парах «запрос — выбранное объявление» из train; добавить в CatBoost
признаки взаимодействия объявления с историей (как часто выбирали объявления этого исполнителя, популярность
заголовка); увеличить пул, который переранжирует CatBoost, — в пуле 94% правильных ответов, а в топ-200 — 91%.